# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process a dataset described by a Croissant schema using the `mlcroissant` library. All dataset entities are referenced by their `@id` identifiers for reproducibility.

### Dataset Source
The dataset Croissant schema is available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's review the available record sets and their fields using their `@id`s.

In [ ]:
# List all record sets and their field IDs in the dataset
record_sets = metadata.record_sets
print("Record sets (`@id`):\n---------------------------")
for rs in record_sets:
    print(f"- @id: {rs.id}\n  name: {rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, data_type: {field.data_type}")

## 3. Data Extraction
We'll extract data from the main record sets. All entities are referenced by their `@id` as shown above.

In [ ]:
# Collect all the record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Pick the first record set for demonstration
main_record_set_id = record_set_ids[0]
print(f"Columns for record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate filtering, normalization, and grouping using fields available in the selected record set.

Replace `<numeric_field_id>` and `<group_field_id>` below with actual IDs after reviewing the columns above (e.g. accession, description, coverage, count, MW, etc).

In [ ]:
# Example field IDs (update according to overview above)
numeric_field_id = None
group_field_id = None
for c in dataframes[main_record_set_id].columns:
    if any(x in c.lower() for x in ["coverage", "count", "mw", "abundance", "value", "number"]):
        numeric_field_id = c
        break
# Search for a categorical field
for c in dataframes[main_record_set_id].columns:
    if any(x in c.lower() for x in ["accession", "sample", "description", "gene", "type"]):
        group_field_id = c
        break

if numeric_field_id is None:
    raise ValueError("Could not find numeric field. Update selection based on dataset.")

df = dataframes[main_record_set_id]
# Try to ensure the numeric field is float for analysis
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = df[numeric_field_id].quantile(0.75)  # Use upper quartile as threshold
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id]].head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and optionally, mean values grouped by the chosen group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(20)
    sns.barplot(x=group_means.values, y=group_means.index, orient='h')
    plt.title(f"Mean of {numeric_field_id} per {group_field_id} (Top 20)")
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(group_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we successfully loaded a complex experimental dataset described by a Croissant schema, explored its record sets referenced by `@id`, and performed basic data filtering, normalization, grouping, and visualization. For further analysis, consult each field's Croissant documentation and investigate additional record sets or metadata provided by the dataset authors.